In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_excel('your data.xlsx', sheet_name='target sheet name')
df = df.dropna(axis=1)
X = df.drop(['Refolding'], axis = 1)
y = df['Refolding']

In [ ]:
from ChemUtils import GlobalStandardScaler
xscaler = GlobalStandardScaler()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
X_train = xscaler.fit_transform(X_train)
X_test = xscaler.transform(X_test)

In [ ]:
print (X_train.std())
print (X_train.mean())
print (X_train.max())

In [ ]:
yscaler = GlobalStandardScaler()
y_train = yscaler.fit_transform(y_train)
y_test = yscaler.transform(y_test)

In [ ]:
def dataaugment(x, betashift = 0.05, slopeshift = 0.05,multishift = 0.05):
    #Shift of baseline
    #calculate arrays
    beta = np.random.random(size=(x.shape[0],1))*2*betashift-betashift
    slope = np.random.random(size=(x.shape[0],1))*2*slopeshift-slopeshift + 1
    #Calculate relative position
    axis = np.array(range(x.shape[1]))/float(x.shape[1])
    #Calculate offset to be added
    offset = slope*(axis) + beta - axis - slope/2. + 0.5

    #Multiplicative
    multi = np.random.random(size=(x.shape[0],1))*2*multishift-multishift + 1

    x = multi*x + offset

    return x


In [ ]:
X = X_train[0:1]
X = np.repeat(X, repeats=10, axis=0)
X_aug = dataaugment(X,betashift = 0.5, slopeshift = 0.5,multishift = 0.5)
    
plt.plot(X_aug.T)
_= plt.plot(X.T, lw=5, c='b')

In [ ]:
shift = np.std(X_train)*0.1
shift

In [ ]:
X_train_aug = np.repeat(X_train, repeats=100, axis=0)
X_train_aug = dataaugment(X_train_aug, betashift = shift, slopeshift = 0.05, multishift = shift)

y_train_aug = np.repeat(y_train, repeats=100, axis=0) #y_train is simply repeated


print (len(X_train_aug))
print (len(y_train_aug))
_ = plt.plot(X_train_aug[0:100].T)

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv1D, Reshape, GaussianNoise
from keras.callbacks import ReduceLROnPlateau

In [ ]:
#Hyperparameters for the network
DENSE = 128
DROPOUT = 0.5
C1_K  = 8 #Number of kernels/feature extractors for first layer
C1_S  = 32 #Width of the convolutional mini networks
C2_K  = 16
C2_S  = 32
activation='relu'

input_dim = X_train.shape[1]

#The model
def make_model():
    model = Sequential()
    model.add(GaussianNoise(0.05, input_shape=(input_dim,)))
    model.add(Reshape((input_dim, 1) ))
    model.add(Conv1D(C1_K, (C1_S), activation=activation, padding="same"))
    model.add(Conv1D(C2_K, (C2_S), padding="same", activation=activation))
    model.add(Conv1D(C2_K, (C2_S), padding="same", activation=activation))
    model.add(Conv1D(C2_K, (C2_S), padding="same", activation=activation))
    model.add(Flatten())
    model.add(Dropout(DROPOUT))
    model.add(Dense(DENSE, activation=activation))
    model.add(Dense(1, activation='linear'))

    model.compile(loss='mse', optimizer=keras.optimizers.Adadelta(lr=0.001))

    return model

In [ ]:
model = make_model()
print(model.summary())

In [ ]:
rdlr = ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=1)

h = model.fit(X_train_aug, y_train_aug, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[rdlr])

In [ ]:
plt.plot(h.history['loss'], label='loss')
plt.plot(h.history['val_loss'], label='val_loss')

plt.yscale('log')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
ax2 = plt.gca().twinx()
ax2.plot(h.history['lr'], color='r')
ax2.set_ylabel('lr',color='r')

plt.legend()

In [ ]:
plt.scatter(y_train, model.predict(X_train))
plt.scatter(y_test, model.predict(X_test))
plt.plot([-2,3],[-2,3])

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
def huber(y_true, y_pred, delta=1.0):
	y_true = y_true.reshape(-1,1)
	y_pred = y_pred.reshape(-1,1)
	return np.mean(delta**2*( (1+((y_true-y_pred)/delta)**2)**0.5 -1))

In [ ]:
def benchmark(X_train,y_train,X_test, y_test, model):
    rmse = np.mean((y_train - model.predict(X_train).reshape(y_train.shape))**2)**0.5
    rmse_test = np.mean((y_test - model.predict(X_test).reshape(y_test.shape))**2)**0.5
    hub = huber(y_train, model.predict(X_train))
    hub_test = huber(y_test, model.predict(X_test))
    print ("RMSE  Train/Test\t%0.2F\t%0.2F"%(rmse, rmse_test))
    print ("Huber Train/Test\t%0.4F\t%0.4F"%(hub, hub_test))

In [ ]:
benchmark(X_train, y_train, X_test, y_test, model)

In [ ]:
y_pred = yscaler.inverse_transform(y_pred)
y_test = yscaler.inverse_transform(y_test)

In [ ]:
y_pred

In [ ]:
y_test

In [ ]:
pred = np.array(y_pred)
test = np.array(y_test)

In [ ]:
import seaborn as sns
fig3 = plt.figure()
sns.regplot(x= test, y=pred, marker = "o", color="darkred", line_kws={"color":"lightblue","alpha":0.5,"lw":4})
plt.title('Actual vs Predicted')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.show()
#fig3.savefig('PvsO.svg', format='svg', dpi=1200)


In [ ]:
x = list(range(len(test)))
fig4 = plt.figure()
plt.scatter(x, test, color="lightblue", label="Actual")
plt.plot(x, pred, color="darkred", label="Predicted")
plt.title('Comparsion plot')
plt.xlabel('Spectra')
plt.ylabel('Conc')
plt.legend()
plt.show() 
#fig4.savefig('comp.svg', format='svg', dpi=1200)

In [ ]:
import sklearn.metrics as metrics

mae = metrics.mean_absolute_error(test, pred)
mse = metrics.mean_squared_error(test, pred)
rmse = np.sqrt(mse) # or mse**(0.5)  
r2 = metrics.r2_score(test,pred)
#t = stdev (test)
#nrmse = rmse/t

print("Results:")
print("MAE:",mae)
print("MSE:", mse)
print("RMSE:", rmse)
#print("NRMSE:", nrmse)
print ("r2:", r2)

#nrmse = rmse/t

In [ ]:
y_pred

In [ ]:
y_test